# RAG Generation with OpenAI

Retrieves relevant chunks from ChromaDB (using `BSC-LT/MrBERT-es` embeddings) and sends them to an OpenAI LLM to generate a grounded answer.

**Prerequisites:**
- Run `ingestion_preprocessing_embedding.ipynb` first so `chroma_db/` exists.
- Copy `../.env.example` to `../.env` and fill in your `OPENAI_API_KEY`.

In [1]:
!pip install chromadb transformers torch openai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load environment variables

In [2]:
import os
from dotenv import load_dotenv

# .env lives one level up (project root), relative to this notebook
load_dotenv(dotenv_path="../.env")

OPENAI_API_KEY    = os.environ["OPENAI_API_KEY"]
EMBEDDING_MODEL   = os.getenv("EMBEDDING_MODEL", "BSC-LT/MrBERT-es")
CHROMA_PATH       = os.getenv("CHROMA_PATH", "chroma_db")
CHROMA_COLLECTION = os.getenv("CHROMA_COLLECTION", "actas_obra")

print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"ChromaDB path   : {CHROMA_PATH}")
print(f"Collection      : {CHROMA_COLLECTION}")
print(f"OpenAI key      : {OPENAI_API_KEY[:8]}...")

Embedding model : BSC-LT/MrBERT-es
ChromaDB path   : chroma_db
Collection      : actas_obra
OpenAI key      : sk-proj-...


## 2. Load embedding model and ChromaDB collection

In [3]:
import chromadb
from transformers import AutoTokenizer, AutoModel
import torch

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(CHROMA_COLLECTION)
print(f"Collection loaded with {collection.count()} chunks.")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection loaded with 662 chunks.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embed_model = AutoModel.from_pretrained(EMBEDDING_MODEL)

def embed_query(text: str) -> list[float]:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = embed_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 18275.24it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. Configure OpenAI client

In [5]:
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

## 4. RAG pipeline

In [6]:
SYSTEM_PROMPT = (
    "Eres un asistente experto en actas de obra de construcción. "
    "Responde ÚNICAMENTE usando la información del contexto proporcionado. "
    "Si la respuesta no está en el contexto, indícalo explícitamente. "
    "Cita el chunk_id de las fuentes que uses cuando sea posible."
)

def retrieve(query: str, n_results: int = 5) -> tuple[list[str], list[dict]]:
    query_embedding = embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results["documents"][0], results["metadatas"][0]

def generate(query: str, docs: list[str], model: str = "gpt-4o-mini") -> str:
    context = "\n\n---\n\n".join(docs)
    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Contexto:\n{context}\n\nPregunta: {query}"}
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

def rag(query: str, n_results: int = 5, model: str = "gpt-4o-mini"):
    docs, metas = retrieve(query, n_results)
    answer = generate(query, docs, model)
    return answer, docs, metas

## 5. Run a query

In [7]:
query = "¿Cuál es el estado actual de la gestión del informe de conexión de acometida pluvial con Drenajes Besós?"

answer, docs, metas = rag(query)

print("=== ANSWER ===")
print(answer)

print("\n=== RETRIEVED CHUNKS ===")
for doc, meta in zip(docs, metas):
    print(f"[chunk_id={meta['chunk_id']}] {doc[:200]}...")
    print()

=== ANSWER ===
El estado actual de la gestión del informe de conexión de acometida pluvial con Drenajes Besós es que se ha recibido un informe favorable con relación al valor del 2%. Esto se menciona como un tema pendiente de otras reuniones (chunk_id: 3).

=== RETRIEVED CHUNKS ===
[chunk_id=404] Por lo tanto, se descartaba el material de PE que proponía la constructora. Renfe comunica tras consultar con el dpto. de Autoprotección que el material de la instalación de la columna seca debe ser a...

[chunk_id=200] La máquina está colgada mediante varillas roscadas, número de unidades:4. La máquina descansa parcialmente sobre perfiles de falso techo, que NO están diseñados para soportar peso (solo sujetan placas...

[chunk_id=441] El 06 de marzo se recibe la planificación actualizada de la constructora, que se adjunta como anejo de esta acta. 3.11. PPI estructura metálica Se solicita a la Constructora la entrega de los PPI (Pla...

[chunk_id=345] Se está realizando el armado de muro de an